# Bronze Layer
## Load data

In [0]:
# Create schema to Adult Income Raw (Bronze Layer)
spark.sql("""
CREATE SCHEMA IF NOT EXISTS adult_income.adult_income_raw
""")

In [0]:
from sklearn.datasets import fetch_openml

# 1. Carregar dataset Adult Income
adult = fetch_openml(name='adult', version=2, as_frame=True)
df = adult.frame.copy()

# 2. Convert pandas DataFrame to Spark DataFrame
df_bronze = spark.createDataFrame(df)

# 3. Write the transformed data to the Bronze layer
bronze_data_path = "adult_income.adult_income_raw.ad_inc_raw"
df_bronze.write.format("delta").mode("overwrite").saveAsTable(bronze_data_path)

# Register the DataFrame as a temporary view so we can run SQL queries
df_bronze.createOrReplaceTempView("ad_inc_raw")

# Example SQL query on the temporary view
result_df = spark.sql("SELECT COUNT(*) as trip_count FROM ad_inc_raw")

# Silver Layer
## Data Preparation

In [0]:
# Create schema to Adult Income Raw (Bronze Layer)
spark.sql("""
CREATE SCHEMA IF NOT EXISTS adult_income.adult_income_silver
""")

In [0]:
# Carregar dados da camada Bronze (ad_inc_raw)
df_raw = spark.table("adult_income.adult_income_raw.ad_inc_raw").toPandas()

# Convert string columns to categorical dtype
for col in ['workclass', 'occupation', 'native-country']:
    df_raw[col] = df_raw[col].astype('category')

# 1) Adicionar a categoria 'Unemployed' apenas se ela ainda não existir
if 'Unemployed' not in df_raw['workclass'].cat.categories:
    df_raw['workclass'] = df_raw['workclass'].cat.add_categories(['Unemployed'])

# 1.1) Preencher os valores nulos
df_raw['workclass'] = df_raw['workclass'].fillna('Unemployed')

# 2) Tratamento de missing values para 'occupation' e 'native-country'
for col in ['occupation', 'native-country']:
    if 'missing' not in df_raw[col].cat.categories:
        df_raw[col] = df_raw[col].cat.add_categories(['missing'])
    df_raw[col] = df_raw[col].fillna('missing')

# Convert pandas DataFrame to Spark DataFrame
df_silver = spark.createDataFrame(df_raw)

# Write the transformed data to the Silver layer
silver_data_path = "adult_income.adult_income_silver.ad_inc_silver"
df_silver.write.format("delta").mode("overwrite").saveAsTable(silver_data_path)

# Register the DataFrame as a temporary view so we can run SQL queries
df_silver.createOrReplaceTempView("ad_inc_silver")

print("Silver layer processing completed.")

# Gold Layer
## Feature Engineering

In [0]:
# Create schema to Adult Income Raw (Bronze Layer)
spark.sql("""
CREATE SCHEMA IF NOT EXISTS adult_income.adult_income_gold
""")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, create_map, lit
from itertools import chain

# Load data from Silver layer
df_silver = spark.table("adult_income.adult_income_silver.ad_inc_silver")

# Create feature engineered DataFrame using PySpark
df_feature = df_silver \
    .withColumn('is_capital_gain', when(col('capital-gain') > 0, 1).otherwise(0)) \
    .drop('capital-gain') \
    .withColumn('is_private', when(col('workclass') == 'Private', 1).otherwise(0)) \
    .withColumn('is_not_private', when(col('workclass') != 'Private', 1).otherwise(0)) \
    .withColumn('is_white', when(col('race') == 'White', 1).otherwise(0)) \
    .withColumn('is_not_white', when(col('race') != 'White', 1).otherwise(0)) \
    .withColumn('is_male', when(col('sex') == 'Male', 1).otherwise(0)) \
    .withColumn('is_female', when(col('sex') == 'Female', 1).otherwise(0)) \
    .withColumn('is_occupation_missing', when(col('occupation') == 'missing', 1).otherwise(0)) \
    .withColumn('is_native-country_missing', when(col('native-country') == 'missing', 1).otherwise(0)) \
    .withColumn('is_from_United-States', when(col('native-country') == 'United-States', 1).otherwise(0)) \
    .withColumn('is_capital_loss', when(col('capital-loss') > 0, 1).otherwise(0)) \
    .withColumn('is_married', when(col('marital-status') == 'Married-civ-spouse', 1).otherwise(0)) \
    .withColumn('is_married_and_not', when((col('marital-status') == 'Married-civ-spouse') | (col('marital-status') == 'Never-married'), 1).otherwise(0)) \
    .withColumn('is_married_notmarried_divorced', when((col('marital-status') == 'Married-civ-spouse') | (col('marital-status') == 'Never-married') | (col('marital-status') == 'Divorced'), 1).otherwise(0)) \
    .withColumn('is_husband', when(col('relationship') == 'Husband', 1).otherwise(0)) \
    .withColumn('is_wife', when(col('relationship') == 'Wife', 1).otherwise(0)) \
    .withColumn('is_husb_and_not_in_fam', when((col('relationship') == 'Husband') & (col('relationship') != 'Not-in-family'), 1).otherwise(0)) \
    .withColumn('edu_x_hours', col('education-num') * col('hours-per-week'))

# marital-status: order labeling por conhecimento de domínio
df_feature = df_feature.withColumn('marital_status_ord',
    when(col('marital-status') == 'Never-married', 1)
    .when(col('marital-status') == 'Separated', 2)
    .when(col('marital-status') == 'Divorced', 3)
    .when(col('marital-status') == 'Widowed', 4)
    .when(col('marital-status') == 'Married-spouse-absent', 5)
    .when(col('marital-status') == 'Married-AF-spouse', 6)
    .when(col('marital-status') == 'Married-civ-spouse', 7)
    .otherwise(None)
)

# relationship: order labeling por conhecimento de domínio
df_feature = df_feature.withColumn('relationship_ord',
    when(col('relationship') == 'Own-child', 1)
    .when(col('relationship') == 'Other-relative', 2)
    .when(col('relationship') == 'Unmarried', 3)
    .when(col('relationship') == 'Not-in-family', 4)
    .when(col('relationship') == 'Wife', 5)
    .when(col('relationship') == 'Husband', 6)
    .otherwise(None)
)

# workclass: order labeling por conhecimento de domínio
df_feature = df_feature.withColumn('workclass_ord',
    when(col('workclass') == 'Never-worked', 1)
    .when(col('workclass') == 'Without-pay', 2)
    .when(col('workclass') == 'Unemployed', 3)
    .when(col('workclass') == 'Private', 4)
    .when(col('workclass') == 'Self-emp-not-inc', 5)
    .when(col('workclass') == 'Self-emp-inc', 6)
    .when(col('workclass') == 'Local-gov', 7)
    .when(col('workclass') == 'State-gov', 8)
    .when(col('workclass') == 'Federal-gov', 9)
    .otherwise(None)
)

# occupation: order labeling por conhecimento de domínio
df_feature = df_feature.withColumn('occupation_ord',
    when(col('occupation') == 'Prof-specialty', 1)
    .when(col('occupation') == 'Craft-repair', 2)
    .when(col('occupation') == 'Adm-clerical', 3)
    .when(col('occupation') == 'Exec-managerial', 4)
    .when(col('occupation') == 'Sales', 5)
    .when(col('occupation') == 'Handlers-cleaners', 6)
    .when(col('occupation') == 'Machine-op-inspct', 7)
    .when(col('occupation') == 'Tech-support',8)
    .when(col('occupation') == 'Transport-moving', 9)
    .when(col('occupation') == 'Farming-fishing', 10)
    .when(col('occupation') == 'Protective-serv',11)
    .when(col('occupation') == 'Priv-house-serv', 12)
    .when(col('occupation') == 'Armed-Forces', 13)
    .when(col('occupation') == 'Other-service', 14)
    .when(col('occupation').isNull(), 15)
    .otherwise(None)
)

# occupation: order labeling por conhecimento de domínio
df_feature = df_feature.withColumn('relationship_ord',
    when(col('relationship') == 'Own-child', 1)
    .when(col('relationship') == 'Other-relative',2)
    .when(col('relationship') == 'Unmarried', 3)
    .when(col('relationship') == 'Not-in-family',4)
    .when(col('relationship') == 'Wife', 5)
    .when(col('relationship') == 'Husband', 6)
    .otherwise(None)
)


# occupation: order labeling por conhecimento de domínio
df_feature = df_feature.withColumn('race_ord',
    when(col('race') == 'White', 1)
    .when(col('race') == 'Black', 2)
    .when(col('race') == 'Asian-Pac-Islander', 3)
    .when(col('race') == 'Amer-Indian-Eskimo', 4)
    .when(col('race') == 'Other', 5)
    .otherwise(None)
)

# occupation: order labeling por conhecimento de domínio
df_feature = df_feature.withColumn('sex_ord',
    when(col('sex') == 'Male', 1)
    .when(col('sex') == 'Female',2)
    .otherwise(None)
)

# native-country: order labeling por conhecimento de domínio
df_feature = df_feature.withColumn('native-country_ord',
    when(col('native-country') == 'United-States', 1)
    .when(col('native-country') == 'Mexico', 2)
    .when(col('native-country') == 'Germany', 3)
    .when(col('native-country') == 'Canada', 4)
    .when(col('native-country') == 'United-Kingdom', 5)
    .when(col('native-country') == 'China', 6)
    .when(col('native-country') == 'Japan', 7)
    .when(col('native-country') == 'Philippines', 8)
    .when(col('native-country') == 'Poland', 9)
    .when(col('native-country') == 'Italy', 10)
    .when(col('native-country') == 'Vietnam', 11)
    .when(col('native-country') == 'Cuba', 12)
    .when(col('native-country') == 'Ireland', 13)
    .when(col('native-country') == 'South', 14)
    .when(col('native-country') == 'Puerto-Rico',15)
    .when(col('native-country') == 'Dominican-Republic', 16)
    .when(col('native-country') == 'El-Salvador', 17)
    .when(col('native-country') == 'Guatemala', 18)
    .when(col('native-country') == 'Haiti', 19)
    .when(col('native-country') == 'Nicaragua', 20)
    .when(col('native-country') == 'Peru', 21)
    .when(col('native-country') == 'Trinadad&Tobago', 22)
    .when(col('native-country') == 'Ecuador', 23)
    .when(col('native-country') == 'Honduras', 24)
    .when(col('native-country') == 'Cambodia', 25)
    .when(col('native-country') == 'Jamaica', 26)
    .when(col('native-country') == 'Thailand', 27)
    .when(col('native-country') == 'Laos', 28)
    .when(col('native-country') == 'Yugoslavia', 29)
    .when(col('native-country') == 'Outlying-US(Guam-USVI-etc)', 30)
    .when(col('native-country') == 'Scotland', 31)
    .when(col('native-country') == 'Hong', 32)
    .when(col('native-country') == 'Trinadad&Tobago', 33)
    .when(col('native-country') == 'Greece', 34)
    .when(col('native-country') == 'China', 35)
    .when(col('native-country') == 'Nicaragua', 36)
    .when(col('native-country') == 'Portugal', 37)
    .when(col('native-country') == 'Outlying-US(Guam-USVI-etc)', 38)
    .when(col('native-country') == 'Peru', 39)
    .when(col('native-country') == 'Holand-Netherlands', 40)
    .when(col('native-country').isNull(), 41)
    .otherwise(None)
)

from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, create_map, lit
from itertools import chain

# ... (todo o código já existente até ao workclass_ord/race_ord/sex_ord/native-country_ord) ...

# --- Mapeamento País -> Continente ---
continente_dict = {
    "United-States":"America", "Canada":"America", "Mexico":"America",
    "Puerto-Rico":"America", "Cuba":"America", "Honduras":"America",
    "Jamaica":"America", "Dominican-Republic":"America", "Ecuador":"America",
    "Haiti":"America", "Columbia":"America", "Guatemala":"America",
    "Nicaragua":"America", "El-Salvador":"America", "Peru":"America",
    "Trinadad&Tobago":"America", "Outlying-US(Guam-USVI-etc)":"America",
    "England":"Europa", "Germany":"Europa", "Greece":"Europa", "Italy":"Europa",
    "Poland":"Europa", "Portugal":"Europa", "Ireland":"Europa", "France":"Europa",
    "Hungary":"Europa", "Scotland":"Europa", "Yugoslavia":"Europa",
    "Holand-Netherlands":"Europa", "United-Kingdom":"Europa",
    "Cambodia":"Asia", "India":"Asia", "Japan":"Asia", "South":"Asia",
    "China":"Asia", "Iran":"Asia", "Philippines":"Asia", "Vietnam":"Asia",
    "Laos":"Asia", "Taiwan":"Asia", "Thailand":"Asia", "Hong":"Asia"
}

# --- % segurança (Global Peace Index, aprox.) e PIB per capita em USD mil (IMF, aprox.) ---
seguranca_dict = {
    "United-States":60, "Canada":75, "Mexico":45, "Puerto-Rico":55, "Cuba":55,
    "Honduras":35, "Jamaica":50, "Dominican-Republic":50, "Ecuador":45, "Haiti":25,
    "Columbia":35, "Guatemala":40, "Nicaragua":45, "El-Salvador":35, "Peru":45,
    "Trinadad&Tobago":55, "Outlying-US(Guam-USVI-etc)":60, "England":75, "United-Kingdom":75,
    "Germany":75, "Greece":60, "Italy":65, "Poland":65, "Portugal":78, "Ireland":80,
    "France":65, "Hungary":65, "Scotland":78, "Yugoslavia":50, "Holand-Netherlands":80,
    "Cambodia":55, "India":45, "Japan":75, "South":55, "China":60, "Iran":30,
    "Philippines":45, "Vietnam":60, "Laos":50, "Taiwan":75, "Thailand":60, "Hong":70
}

pib_dict = {
    "United-States":90, "Canada":55, "Mexico":13, "Puerto-Rico":34, "Cuba":9,
    "Honduras":3, "Jamaica":6, "Dominican-Republic":11, "Ecuador":6, "Haiti":1.8,
    "Columbia":7, "Guatemala":5.5, "Nicaragua":2.5, "El-Salvador":5, "Peru":8,
    "Trinadad&Tobago":18, "Outlying-US(Guam-USVI-etc)":35, "England":58, "United-Kingdom":58,
    "Germany":60, "Greece":22, "Italy":39, "Poland":24, "Portugal":30, "Ireland":130,
    "France":45, "Hungary":20, "Scotland":55, "Yugoslavia":12, "Holand-Netherlands":74,
    "Cambodia":2, "India":2.7, "Japan":34, "South":32, "China":13, "Iran":5,
    "Philippines":4, "Vietnam":4.5, "Laos":2, "Taiwan":33, "Thailand":7.5, "Hong":57
}

# Valores default para países ausentes/nulos (ex: "?")
DEFAULT_SEG, DEFAULT_PIB = 50, 20

continente_map = create_map([lit(x) for x in chain(*continente_dict.items())])
seguranca_map = create_map([lit(x) for x in chain(*seguranca_dict.items())])
pib_map = create_map([lit(x) for x in chain(*pib_dict.items())])

df_feature = df_feature \
    .withColumn('continente', F.coalesce(continente_map[col('native-country')], lit('Desconhecido'))) \
    .withColumn('seguranca_pct', F.coalesce(seguranca_map[col('native-country')], lit(DEFAULT_SEG))) \
    .withColumn('pib_percapita', F.coalesce(pib_map[col('native-country')], lit(DEFAULT_PIB))) \
    .withColumn('indice_seg_econ', (col('seguranca_pct') / 100) * col('pib_percapita'))

# Normalizar indice_seg_econ (min-max) e inverter -> risco_pais (0-10)
min_max = df_feature.agg(
    F.min('indice_seg_econ').alias('min_idx'),
    F.max('indice_seg_econ').alias('max_idx')
).collect()[0]
min_idx, max_idx = min_max['min_idx'], min_max['max_idx']

df_feature = df_feature.withColumn(
    'risco_pais',
    10 * (1 - (col('indice_seg_econ') - lit(min_idx)) / (lit(max_idx) - lit(min_idx)))
).withColumn('score_pais', (col('risco_pais') / 10) * 3)  # escalado para 0-3

# Ranking numérico dos continentes (1 = melhor/menos risco)
rank_continente_df = df_feature.groupBy('continente') \
    .agg(F.avg('risco_pais').alias('media_risco')) \
    .withColumn('rank_continente', F.dense_rank().over(
        __import__('pyspark.sql.window', fromlist=['Window']).Window.orderBy('media_risco')
    )).select('continente', 'rank_continente')

df_feature = df_feature.join(rank_continente_df, on='continente', how='left')

# --- Scores individuais (0-3, mesma escala do score_pais) ---
df_feature = df_feature \
    .withColumn('score_idade',
        when(col('age') < 25, 3).when(col('age') < 45, 1).otherwise(0)) \
    .withColumn('score_educacao',
        when(col('education').isin('Preschool','1st-4th','5th-6th'), 3)
        .when(col('education') == 'HS-grad', 2)
        .when(col('education') == 'Some-college', 1)
        .otherwise(0)) \
    .withColumn('score_horas',
        when(col('hours-per-week') < 20, 3).when(col('hours-per-week') < 40, 1).otherwise(0)) \
    .withColumn('score_occupation',
        when(col('occupation').isin('Priv-house-serv','Other-service'), 3)
        .when(col('occupation') == 'Handlers-cleaners', 2)
        .when(col('occupation').isin('Exec-managerial','Prof-specialty'), 0)
        .otherwise(1)) \
    .withColumn('score_workclass',
        when(col('workclass').isin('Never-worked','Without-pay'), 3)
        .when(col('workclass') == 'Self-emp-not-inc', 2)
        .when(col('workclass') == 'Private', 1)
        .otherwise(0))

# --- Score final de risco (numérico contínuo, sem strings, sem income) ---
df_feature = df_feature.withColumn(
    'risco_total',
    col('score_idade') + col('score_educacao') + col('score_horas') +
    col('score_occupation') + col('score_workclass') + col('score_pais')
)

# occupation: order labeling por conhecimento de domínio
df_feature

# Write to Gold layer
gold_data_path = "adult_income.adult_income_gold.ad_inc_gold"
df_feature.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_data_path)

print("Gold layer processing completed.")
print(f"Data saved to: {gold_data_path}")
